## LSTM Power Consumption Prediction

### 1. Import libraries

In [30]:
print("Hello World")

Hello World


In [31]:
import re
from collections import Counter
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
print("PyTorch version:", torch.__version__)

PyTorch version: 2.13.0+cpu


### 2. Import Dataset

In [35]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
individual_household_electric_power_consumption = fetch_ucirepo(id=235) 
  
# data (as pandas dataframes) 
X_raw = individual_household_electric_power_consumption.data.features 

c:\Users\doubl\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\ucimlrepo\fetch.py:97: DtypeWarning: Columns (0: Global_active_power, 1: Global_reactive_power, 2: Voltage, 3: Global_intensity, 4: Sub_metering_1, 5: Sub_metering_2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


### 3. Deal with Missing Values (Clean Dataset)

In [36]:
X = X_raw
X = X.replace("?", np.nan)
X['datetime'] = pd.to_datetime(X["Date"] + " " + X["Time"], format="%d/%m/%Y %H:%M:%S")

X.index = X['datetime']
print("Missing values by column:")
print(X.isna().sum())

cols_to_interpolate = ["Global_active_power", "Sub_metering_1", "Sub_metering_2", "Sub_metering_3"]

# Interpolate missing values in sub_metering_3 (the field with missing values)
for col in cols_to_interpolate:
    X[col] = pd.to_numeric(X[col]).interpolate(method='time')

# Calculate the average energy consumed every minute (in Watt*hours)
X['power'] = (
    X['Global_active_power']*1000/60 - X['Sub_metering_1'] - 
    X['Sub_metering_2'] - X['Sub_metering_3'])

# Resample into hours by taking the average
X_hrs = X['power'].resample("h").mean()

# Split into train and test data
X_train = X_hrs.loc[X_hrs.index < X_hrs.index.max() - pd.DateOffset(months=6)]
X_test = X_hrs.loc[X_hrs.index > X_hrs.index.max() - pd.DateOffset(months=6)]

Missing values by column:
Date                         0
Time                         0
Global_active_power      25979
Global_reactive_power    25979
Voltage                  25979
Global_intensity         25979
Sub_metering_1           25979
Sub_metering_2           25979
Sub_metering_3           25979
datetime                     0
dtype: int64


### 4. Implement LSTM Algorithm

In [ ]:
class SimpleLSTM(nn.Module):
    def __init__(self, embed_dim, hidden_dim, num_layers, output_size):
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_size)

    def forward(self, x, h0=None, c0=None):
        out, (hn, cn) = self.lstm(out, (h0), c0)
        out = self.fc(out)
        return out, (hn, cn)

criterion = nn.MSELoss()
model = SimpleLSTM(input_dim = 1, hidden_dim=64, output_size=1, num_layers=1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


for epoch in range(20):
    model.train()
    optimizer.zero_grad()

    predictions, hn, cn = model(X_train)
    loss = criterion(predictions, y_train)